In [ ]:
# 1. Mount your Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

# 2. Import the operating system library
import os

# 3. Change the directory to your specific folder
# Make sure this matches the exact name of your folder in Drive
folder_path = '/content/drive/MyDrive/Colab/Engine'
os.chdir(folder_path)

# Optional: Verify you are in the right place
print("Current working directory:", os.getcwd())

In [ ]:
import os
import io
import re
import zstandard as zstd
import chess
import chess.pgn
import torch
from tqdm import tqdm

EVAL_REGEX = re.compile(r'\[%eval\s+([^\]]+)\]')

def parseLichessEval(comment: str) -> float | None:
    # Usiamo la regex globale per estrarre il valore
    match = EVAL_REGEX.search(comment)
    if not match: 
        return None
    val_str = match.group(1)
    if '#' in val_str:
        mate_in = int(val_str.replace('#', ''))
        return 15.0 if mate_in > 0 else -15.0
    else:
        return float(val_str)
    
def board_to_indices(board: chess.Board) -> list:
    indices = []
    piece_type_map = {
        chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
        chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5
    }
    
    # Iteriamo SOLO sulle case occupate, ignorando le case vuote
    for square, piece in board.piece_map().items():
        color_offset = 0 if piece.color == chess.WHITE else 6
        idx = (piece_type_map[piece.piece_type] + color_offset) * 64 + square
        indices.append(idx)
        
    padded = indices + [768] * (32 - len(indices))
    return padded

def MinEloPlayer(game: chess.pgn.Game, min_elo: int = 2000) -> bool:
    white_elo_str, black_elo_str = game.headers.get("WhiteElo"), game.headers.get("BlackElo")
    white_elo, black_elo = None, None
    try:
        if white_elo_str and white_elo_str.isdigit():
            white_elo = int(white_elo_str)
        if black_elo_str and black_elo_str.isdigit():
            black_elo = int(black_elo_str)
        if white_elo is None and black_elo is None:
            return False
        if white_elo < min_elo or black_elo < min_elo: 
            return False
        return True
    except (ValueError, IndexError):
        return False

def TimeControlGreaterRapidGameOnly(game: chess.pgn.Game, min_time_sec: int = 600) -> bool:
    tc_str = game.headers.get("TimeControl")
    if not tc_str or tc_str == "-":
        return False
    try:
        base_time_str = tc_str.split('+')[0]
        base_time = int(base_time_str)
        if base_time < min_time_sec: 
            return False
        return True
    except (ValueError, IndexError):
        return False

def StandardAndPlayCountValid(game: chess.pgn.Game) -> bool:
    try:
        variant = game.headers.get("Variant", "Standard")
        if variant != "Standard":
            return False        
        ply_count_str = game.headers.get("PlyCount")
        if ply_count_str and ply_count_str.isdigit():
            if int(ply_count_str) < 10:
                return False
        return True
    except (ValueError, IndexError):
        return False

def HeadersControl(game: chess.pgn.Game | None, min_elo: int = 2000, min_time_sec: int = 600) -> bool:
    if game is None:
        return False
    if not StandardAndPlayCountValid(game):
        return False
    if not TimeControlGreaterRapidGameOnly(game, min_time_sec):
        return False
    if not MinEloPlayer(game, min_elo):
        return False
    return True

def create_dataset(zst_path: str, save_path: str, max_positions: int = 100000, min_elo: int = 2000):
    if not os.path.exists(zst_path):
        print("errore nella locazione pgn")
        return
        
    print("Inizio rasterizzazione dati")
    all_inputs = []
    all_targets = []
    positions_extracted = 0
    games_processed = 0
    
    proc_bar = tqdm(total=max_positions, desc="Posizioni")
    
    with open(zst_path, "rb") as compressed_file:
        dctx = zstd.ZstdDecompressor()
        with dctx.stream_reader(compressed_file) as reader:
            text_stream = io.TextIOWrapper(reader, encoding='utf-8')    
            
            while positions_extracted < max_positions:
                game = chess.pgn.read_game(text_stream)
                
                if game is None: 
                    break 
                    
                if not HeadersControl(game, min_elo, 600):
                    continue
                    
                games_processed += 1
                board = game.board()
                
                for node in game.mainline():
                    board.push(node.move)
                    score = parseLichessEval(node.comment)
                    
                    if score is not None:
                        indices = board_to_indices(board)
                        all_inputs.append(indices)
                        all_targets.append([score])
                        
                        positions_extracted += 1
                        proc_bar.update(1)
                    
                    if positions_extracted >= max_positions:
                        break
                        
    proc_bar.close()
    print(f"partite lette : {games_processed}")
    print("Conversione in tensore")
    
    tensor_inputs = torch.tensor(all_inputs, dtype=torch.long)
    tensor_targets = torch.tensor(all_targets, dtype=torch.float32)
    torch.save((tensor_inputs, tensor_targets), save_path)
    print(f"Dataset salvato come tensore in {save_path}")

if __name__ == "__main__":
    create_dataset(
        zst_path="partite/lichess_db_standard_rated_2026-01.pgn.zst",
        save_path="dataset5M.pt",
        max_positions=5000000,
        min_elo=2000
    )

Inizio rasterizzazione dati


Posizioni:   1%|          | 57744/5000000 [31:06<50:36:21, 27.13it/s] 

KeyboardInterrupt: 